# 🎙️ Pipeline Fallback Step: Audio Splitting: Pydub Waveform Slicing

Splits and concatenates audio turns belonging to the same speaker using the Pydub AudioSegment library.

## Environment Setup

In [ ]:
%%capture
!pip install pydub
from pydub import AudioSegment
import json
import os

## Google Drive Mount & Form Configuration

In [ ]:
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# @markdown ### 📂 Splitting Configuration
input_audio_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/standardized.wav" # @param {type:"string"}
timeline_json_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/maraulikhurad3_refined_timeline.json" # @param {type:"string"}
output_dir = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/MarauliKhurad3/isolated_speakers/" # @param {type:"string"}

os.makedirs(output_dir, exist_ok=True)

## Execute Fallback Processing

In [ ]:
try:
    with open(timeline_json_path, "r", encoding="utf-8") as f:
        segments = json.load(f)
        
    def parse_time_ms(time_str, idx):
        part = time_str.split('-')[idx].replace('[','').replace(']','').strip()
        parts = part.split(':')
        return int((float(parts[-2])*60 + float(parts[-1])) * 1000)

    print(f"Loading full audio file via Pydub: {input_audio_path}...")
    audio = AudioSegment.from_file(input_audio_path)
    
    os.makedirs(output_dir, exist_ok=True)
    speaker_audio = {}
    
    for entry in segments:
        start_ms = parse_time_ms(entry['time'], 0)
        end_ms = parse_time_ms(entry['time'], 1)
        speaker = entry['speaker']
        
        chunk = audio[start_ms:end_ms]
        if speaker not in speaker_audio:
            speaker_audio[speaker] = chunk
        else:
            speaker_audio[speaker] += chunk
            
    for speaker, spk_chunk in speaker_audio.items():
        out_file = os.path.join(output_dir, f"{speaker}_isolated.wav")
        spk_chunk.export(out_file, format="wav", codec="pcm_s16le")
        print(f"Exported speaker isolated file: {out_file}")
        
    print("\n[SUCCESS] Speaker isolation complete!")
except Exception as e:
    print(f"\n[ERROR] Pydub splitting failed: {e}")